# Investigating the `pypi_index.filename` UNIQUE conflict on db2 backfill

`db2_backfill.py`'s `migrate_wheel()` step logs a conflict where the same wheel filename
(`apicore_python-1.0.0-py3-none-any.whl`, `apicore_python-1.0.1-py3-none-any.whl`) appears
in `v.db.wheel` under two different literal `project` spellings: `APICORE_Python` and
`APICORE-Python`.

A prior investigation confirmed via the live PyPI Simple API that these are **not** two
distinct upstream projects -- `APICORE-Python` and `APICORE_Python` both resolve (via PEP 503
normalization) to the exact same canonical project `apicore-python`, with one shared release
history.

Open questions this notebook investigates:

1. Does `v.db.project` (the crawler's per-project index/bookmark table) contain **two** rows
   for this project (one per name spelling), or just one?
2. Does `v.db.wheel` contain rows for **both** spellings, and if so, how do their timestamps
   line up?
3. Does the *real* PyPI root Simple index (`https://pypi.org/simple/`) actually list two
   separate anchor entries for this project, or does it emit is a single normalized entry --
   i.e. did the duplicate spelling originate upstream (from PyPI) or downstream (in our own
   crawl/reroll logic)?

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("/Users/anil/code/reroll-data/data/v.db")
assert DB_PATH.exists(), DB_PATH

# Read-only connection -- never mutate v.db.
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
con.row_factory = sqlite3.Row


def q(sql, params=()):
    return pd.read_sql_query(sql, con, params=params)


q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")

,name
0,meta
1,metadata_blob
2,project
3,repodata_conversion
4,wheel
5,wheel_metadata


In [2]:
q("""
SELECT name, index_serial, crawled_serial, status, n_wheels, error, fetched_at
  FROM project
 WHERE name IN ('APICORE-Python', 'APICORE_Python', 'apicore-python', 'apicore_python')
    OR lower(replace(replace(name,'_','-'),'.','-')) = 'apicore-python'
""")

,name,index_serial,crawled_serial,status,n_wheels,error,fetched_at
0,APICORE-Python,30545251,30545251,gone,2,None,1785559004
1,APICORE_Python,39625290,39625290,done,3,None,1786003027


In [3]:
q("""
SELECT project, filename, size, upload_time, first_seen, last_seen, yanked
  FROM wheel
 WHERE project IN ('APICORE-Python', 'APICORE_Python')
 ORDER BY project, filename
""")

,project,filename,size,upload_time,first_seen,last_seen,yanked
0,APICORE-Python,apicore_python-1.0.0-py3-none-any.whl,5919,2025-07-25T17:43:12.233677+00:00,1785558999,1785558999,0
1,APICORE-Python,apicore_python-1.0.1-py3-none-any.whl,5920,2025-08-06T08:27:45.403464+00:00,1785558999,1785558999,0
2,APICORE_Python,apicore_python-1.0.0-py3-none-any.whl,5919,2025-07-25T17:43:12.233677+00:00,1786003023,1786003023,0
3,APICORE_Python,apicore_python-1.0.1-py3-none-any.whl,5920,2025-08-06T08:27:45.403464+00:00,1786003023,1786003023,0
4,APICORE_Python,apicore_python-2.1.0-py3-none-any.whl,16290,2026-08-02T19:08:05.571807+00:00,1786003023,1786003023,0


In [8]:
import requests

# Root index fetched exactly the way crawl.py's refresh_index() does: JSON-only Accept.
# Stream it and search for the literal "apicore" substring (case-insensitive) rather than
# loading the whole multi-hundred-MB payload into memory / context.
url = "https://pypi.org/simple/"
headers = {
    "Accept": "application/vnd.pypi.simple.v1+json",
    "User-Agent": "reroll-data-investigation/0.1",
}

found = []
buf = ""
CTX = 120
with requests.get(url, headers=headers, stream=True, timeout=180) as resp:
    resp.raise_for_status()
    for chunk in resp.iter_content(chunk_size=1 << 20, decode_unicode=False):
        chunk = chunk.decode("utf-8", errors="ignore")
        buf += chunk
        # Keep buffer bounded but with overlap so we don't miss a match split across chunks.
        if "apicore" in buf.lower():
            idx = 0
            low = buf.lower()
            while True:
                i = low.find("apicore", idx)
                if i == -1:
                    break
                found.append(buf[max(0, i - 5) : i + 40])
                idx = i + 1
        if len(buf) > 2_000_000:
            buf = buf[-200:]  # keep a small tail for cross-chunk matches

print(f"total matches: {len(found)}")
for f in found:
    print(repr(f))

total matches: 10
'me":"apicore"},{"_last-serial":39625290,"name'
'me":"APICORE_Python"},{"_last-serial":1470292'
'":"doapicore"},{"_last-serial":35678431,"name'
'iten-apicore"},{"_last-serial":10462555,"name'
'-mcp-apicore"},{"_last-serial":31329976,"name'
'-mcp-apicore"},{"_last-serial":31329976,"name'
'e":"sapicore"},{"_last-serial":12015057,"name'
'e":"sapicore"},{"_last-serial":12015057,"name'
':"tg-apicore"},{"_last-serial":18771995,"name'
':"tg-apicore"},{"_last-serial":18771995,"name'


In [9]:
import requests

url = "https://pypi.org/simple/"
headers = {
    "Accept": "application/vnd.pypi.simple.v1+json",
    "User-Agent": "reroll-data-investigation/0.1",
}

CTX = 150
found = []
buf = ""
with requests.get(url, headers=headers, stream=True, timeout=180) as resp:
    resp.raise_for_status()
    for chunk in resp.iter_content(chunk_size=1 << 20, decode_unicode=False):
        chunk = chunk.decode("utf-8", errors="ignore")
        buf += chunk
        low = buf.lower()
        idx = 0
        while True:
            i = low.find("apicore", idx)
            if i == -1:
                break
            found.append(buf[max(0, i - CTX) : i + CTX])
            idx = i + 1
        if len(buf) > 2_000_000:
            buf = buf[-CTX - 10 :]

print(f"total matches: {len(found)}")
for f in found:
    print("----")
    print(f)

total matches: 10
----
umers"},{"_last-serial":39445546,"name":"api-contract-guard"},{"_last-serial":36727379,"name":"api-contract-tester"},{"_last-ser
ial":30219564,"name":"apicore"},{"_last-serial":39625290,"name":"APICORE_Python"},{"_last-serial":14702927,"name":"api-correios"
},{"_last-serial":35605663,"name":"apicost"}
----
i-contract-guard"},{"_last-serial":36727379,"name":"api-contract-tester"},{"_last-serial":30219564,"name":"apicore"},{"_last-ser
ial":39625290,"name":"APICORE_Python"},{"_last-serial":14702927,"name":"api-correios"},{"_last-serial":35605663,"name":"apicost"
},{"_last-serial":8888356,"name":"apicount"}
----
89,"name":"doapfiend-html"},{"_last-serial":791290,"name":"doapfiend-vcs"},{"_last-serial":2313755,"name":"doapi"},{"_last-seria
l":25773536,"name":"doapicore"},{"_last-serial":35678431,"name":"do-app-sandbox"},{"_last-serial":9231203,"name":"DOAP-TimeUtils
"},{"_last-serial":60722,"name":"DoapViewPlu
----
e":"HiTech-STT"},{"_last-serial":38201626,"name":"hit

## Scope: how many other projects have this same pattern?

`project.name` is the raw literal string PyPI's root index reported at crawl time (see
`crawl.py::refresh_index`, which inserts `entry["name"]` verbatim, keyed by `name` with no
PEP 503 normalization). If a project's upstream raw name spelling changes (case, or
`-`/`_`/`.` swap) between two crawls, our crawler treats it as a brand new project row
rather than a rename -- exactly what happened with `APICORE-Python` -> `APICORE_Python`.

Group `project` by its PEP 503-normalized name and look for groups with more than one raw
spelling to size up how common this is.

In [10]:
import re

NAME_RUNS = re.compile(r"[-_.]+")


def normalize(name: str) -> str:
    return NAME_RUNS.sub("-", name).lower()


con.create_function("pep503_normalize", 1, normalize)

dupe_groups = q("""
SELECT pep503_normalize(name) AS norm_name,
       count(*) AS n_spellings,
       group_concat(name, ' | ') AS spellings,
       group_concat(status, ' | ') AS statuses,
       sum(n_wheels) AS total_wheel_rows_across_spellings
  FROM project
 GROUP BY norm_name
HAVING count(*) > 1
 ORDER BY total_wheel_rows_across_spellings DESC
""")
print(f"normalized-name groups with >1 raw spelling: {len(dupe_groups)}")
dupe_groups.head(20)

normalized-name groups with >1 raw spelling: 53


,norm_name,n_spellings,spellings,statuses,total_wheel_rows_across_spellings
0,llm-runner,2,llm-runner | llm_runner,gone | done,1931
1,tgcrypto2,2,TgCrypto2 | tgcrypto2,gone | done,990
2,btclib-libsecp256k1,2,btclib_libsecp256k1 | btclib-libsecp256k1,gone | done,329
3,pyrvc,2,PyRVC | pyrvc,gone | done,310
4,typhoontest,2,TyphoonTest | typhoontest,gone,287
5,single-cell-metabolomics,2,single-cell-metabolomics | single_cell_metabol...,gone | done,208
6,pymupdfpro,2,PyMuPDFPro | pymupdfpro,gone,123
7,qldpc,2,qLDPC | qldpc,gone | done,100
8,office365-rest-python-client,2,Office365-REST-Python-Client | office365-rest-...,gone | done,89
9,isa-model,2,isa-model | isa_model,gone | done,67


In [11]:
pd.set_option("display.max_colwidth", 60)
dupe_groups

,norm_name,n_spellings,spellings,statuses,total_wheel_rows_across_spellings
0,llm-runner,2,llm-runner | llm_runner,gone | done,1931
1,tgcrypto2,2,TgCrypto2 | tgcrypto2,gone | done,990
2,btclib-libsecp256k1,2,btclib_libsecp256k1 | btclib-libsecp256k1,gone | done,329
3,pyrvc,2,PyRVC | pyrvc,gone | done,310
4,typhoontest,2,TyphoonTest | typhoontest,gone,287
5,single-cell-metabolomics,2,single-cell-metabolomics | single_cell_metabolomics,gone | done,208
6,pymupdfpro,2,PyMuPDFPro | pymupdfpro,gone,123
7,qldpc,2,qLDPC | qldpc,gone | done,100
8,office365-rest-python-client,2,Office365-REST-Python-Client | office365-rest-python-client,gone | done,89
9,isa-model,2,isa-model | isa_model,gone | done,67


In [12]:
# For each dupe group's raw-name pair, how many *wheel filenames* actually collide
# (i.e. would trip pypi_index's UNIQUE(filename) during the db2 backfill)?
norm_names = dupe_groups["norm_name"].tolist()

rows = []
for norm in norm_names:
    spellings = q("SELECT name FROM project WHERE pep503_normalize(name) = ?", (norm,))[
        "name"
    ].tolist()
    if len(spellings) != 2:
        rows.append((norm, spellings, None, None, None))
        continue
    a, b = spellings
    fa = set(q("SELECT filename FROM wheel WHERE project = ?", (a,))["filename"])
    fb = set(q("SELECT filename FROM wheel WHERE project = ?", (b,))["filename"])
    rows.append((norm, f"{a} | {b}", len(fa), len(fb), len(fa & fb)))

overlap_df = pd.DataFrame(
    rows,
    columns=[
        "norm_name",
        "spellings",
        "n_wheels_a",
        "n_wheels_b",
        "n_colliding_filenames",
    ],
)
print(
    "groups with >=1 colliding filename (i.e. will hit db2's UNIQUE(filename)):",
    (overlap_df["n_colliding_filenames"].fillna(0) > 0).sum(),
    "/",
    len(overlap_df),
)
overlap_df.sort_values("n_colliding_filenames", ascending=False)

groups with >=1 colliding filename (i.e. will hit db2's UNIQUE(filename)): 33 / 53


,norm_name,spellings,n_wheels_a,n_wheels_b,n_colliding_filenames
0,llm-runner,llm-runner | llm_runner,953,978,953
1,tgcrypto2,TgCrypto2 | tgcrypto2,474,516,474
3,pyrvc,PyRVC | pyrvc,148,162,148
5,single-cell-metabolomics,single-cell-metabolomics | single_cell_metabolomics,103,105,103
2,btclib-libsecp256k1,btclib-libsecp256k1 | btclib_libsecp256k1,252,77,77
7,qldpc,qLDPC | qldpc,50,50,50
8,office365-rest-python-client,Office365-REST-Python-Client | office365-rest-python-client,44,45,44
9,isa-model,isa-model | isa_model,33,34,33
10,dashai,DashAI | dashAI,32,34,32
12,localassistant,LocalAssistant | localassistant,18,20,18


In [13]:
total_colliding = overlap_df["n_colliding_filenames"].fillna(0).sum()
total_wheel_rows_in_dupe_groups = (
    overlap_df["n_wheels_a"].fillna(0).sum() + overlap_df["n_wheels_b"].fillna(0).sum()
)
print(f"53 normalized-name groups have 2 raw spellings in v.db.project")
print(f"33 of those groups have >=1 wheel filename shared between both spellings")
print(f"total colliding wheel filenames across the corpus: {int(total_colliding):,}")
print(
    f"total v.db.wheel rows living under *either* spelling in these 53 groups: {int(total_wheel_rows_in_dupe_groups):,}"
)

53 normalized-name groups have 2 raw spellings in v.db.project
33 of those groups have >=1 wheel filename shared between both spellings
total colliding wheel filenames across the corpus: 2,020
total v.db.wheel rows living under *either* spelling in these 53 groups: 4,827


## Conclusion

### How the duplicate got into `v.db` in the first place

1. **`v.db.project` is the "index" table** -- one row per project *as literally reported by
   PyPI's root `/simple/` index* (`crawl.py::refresh_index`). It inserts `entry["name"]`
   verbatim as the primary key, with **no PEP 503 normalization**.
2. **PyPI's root index legitimately reports a raw, non-normalized, mutable display name** per
   PEP 691 -- confirmed by streaming the live 40+ MB root index just now: the current entry for
   this project is exactly `{"_last-serial":39625290,"name":"APICORE_Python"}`. There is no
   `APICORE-Python` entry live today.
3. `v.db.project` shows **both** entries: the old `APICORE-Python` row (`index_serial=30545251`,
   `status='gone'`) and the current `APICORE_Python` row (`index_serial=39625290`,
   `status='done'`). `status='gone'` is `refresh_index()`'s own reconciliation marker -- it means
   a *later* full-index refresh no longer saw that literal string in PyPI's project list, because
   PyPI itself had, in the interim, started reporting the raw name as `APICORE_Python` instead
   (almost certainly triggered by the `2.1.0` release re-declaring `Name: APICORE_Python` in its
   metadata, which Warehouse uses to refresh the project's raw display name).
4. Because `project.name` (raw, un-normalized) is the crawler's identity key, that upstream
   rename was never recognized as "the same project, new spelling." It was crawled as a **brand
   new project**, fully re-fetching `1.0.0` and `1.0.1`'s wheel listings a second time under the
   new key, while the old key's rows (and their wheel rows) were left in place, just flagged
   `gone`. Nothing merges or dedupes `gone` project rows against their normalized twin -- that
   step doesn't exist anywhere in this codebase today.
5. **`v.db.wheel` inherits the same duplication**: 2 rows under `APICORE-Python` (`1.0.0`,
   `1.0.1`) and 3 rows under `APICORE_Python` (`1.0.0`, `1.0.1`, `2.1.0`) -- 5 rows total for
   what is, upstream, a single 3-release project.
6. This is not an isolated incident: **53 normalized-name groups** in the current `v.db.project`
   carry two raw spellings, **33 of them** produce at least one colliding wheel `filename`
   (**2,020 colliding filenames total**, out of **4,827** `wheel` rows living under one of these
   dupe-spelling projects).

### Bottom line
The bug is entirely on our side, not PyPI's: `v.db.project`/`v.db.wheel` key on the *raw*
project name, but PyPI's raw display name for a project is not a stable identity -- only the
PEP 503-*normalized* name is. `pypi_index.filename` being globally UNIQUE in the new `db2`
schema is correct; it's `v.db`'s own historical `(project, filename)` key that was too loose,
and it let ~2,020 rows of pure duplication accumulate silently until the backfill's stricter
constraint surfaced it.

## Would dropping `status='gone'` rows fix this?

Test the hypothesis: is `status='gone'` a safe proxy for "this row is the stale half of a
rename duplicate", or does it also cover unrelated cases (projects genuinely deleted from
PyPI entirely, with no normalized twin at all)? If the latter, blanket-dropping `gone` rows
would destroy real historical data that has nothing to do with this bug.

In [14]:
total_projects = q("SELECT count(*) AS n FROM project")["n"][0]
gone_projects = q("SELECT count(*) AS n FROM project WHERE status = 'gone'")["n"][0]
done_projects = q("SELECT count(*) AS n FROM project WHERE status = 'done'")["n"][0]
other_status = q(
    "SELECT status, count(*) AS n FROM project WHERE status NOT IN ('gone','done') OR status IS NULL GROUP BY status"
)

print(f"total project rows:  {total_projects:,}")
print(f"status = 'gone':     {gone_projects:,}")
print(f"status = 'done':     {done_projects:,}")
print("other statuses:")
other_status

total project rows:  867,071
status = 'gone':     248
status = 'done':     866,466
other statuses:


,status,n
0,None,357


In [15]:
# Of the 248 'gone' rows, how many actually belong to one of the 53 rename-duplicate
# groups (i.e. have a normalized twin at all) vs are standalone (project genuinely
# deleted from PyPI, never renamed, no duplicate)?
gone_with_twin = q("""
WITH norm AS (
    SELECT name, status, pep503_normalize(name) AS norm_name FROM project
),
twin_counts AS (
    SELECT norm_name, count(*) AS n FROM norm GROUP BY norm_name
)
SELECT n.status, count(*) AS n_rows
  FROM norm n
  JOIN twin_counts t USING (norm_name)
 WHERE n.status = 'gone'
 GROUP BY (t.n > 1)
""")

gone_in_dupe_group = q("""
WITH norm AS (
    SELECT name, status, pep503_normalize(name) AS norm_name FROM project
),
twin_counts AS (
    SELECT norm_name, count(*) AS n FROM norm GROUP BY norm_name
)
SELECT
    sum(CASE WHEN t.n > 1 THEN 1 ELSE 0 END) AS gone_with_twin,
    sum(CASE WHEN t.n = 1 THEN 1 ELSE 0 END) AS gone_standalone_no_twin
  FROM norm n
  JOIN twin_counts t USING (norm_name)
 WHERE n.status = 'gone'
""")
gone_in_dupe_group

,gone_with_twin,gone_standalone_no_twin
0,53,195


In [17]:
# Materialize normalized names into an indexed temp table so the self-join isn't O(n^2).
con.execute("DROP TABLE IF EXISTS temp._proj_norm")
con.execute("""
    CREATE TEMP TABLE _proj_norm AS
    SELECT name, status, pep503_normalize(name) AS norm_name FROM project
""")
con.execute("CREATE INDEX temp_proj_norm_idx ON _proj_norm(norm_name)")
con.commit()

breakdown = q("""
SELECT a.name AS gone_name, b.name AS twin_name, b.status AS twin_status
  FROM _proj_norm a
  JOIN _proj_norm b ON a.norm_name = b.norm_name AND a.name <> b.name
 WHERE a.status = 'gone'
 ORDER BY twin_status, gone_name
""")
print(breakdown["twin_status"].value_counts())
breakdown[breakdown["twin_status"] == "gone"]

twin_status
done    50
Name: count, dtype: int64


,gone_name,twin_name,twin_status


In [18]:
# The "3 non gone|done groups" mystery: group_concat() silently skips NULL values,
# so a "gone + NULL-status" pair prints as bare "gone" in the earlier summary.
null_status_twins = q("""
SELECT a.name AS gone_name, a.status AS a_status, b.name AS twin_name, b.status AS twin_status
  FROM _proj_norm a
  JOIN _proj_norm b ON a.norm_name = b.norm_name AND a.name <> b.name
 WHERE a.status = 'gone' AND b.status IS NULL
""")
null_status_twins

,gone_name,a_status,twin_name,twin_status
0,PyMuPDFPro,gone,pymupdfpro,None
1,Typhoon-HIL-API,gone,typhoon-hil-api,None
2,TyphoonTest,gone,typhoontest,None


In [19]:
# Confirm: do the 3 "gone + not-yet-crawled twin" pairs have any wheel rows on the
# not-yet-crawled side at all? (status IS NULL should mean "discovered, not crawled yet".)
for name in null_status_twins["twin_name"]:
    n = q("SELECT count(*) AS n FROM wheel WHERE project = ?", (name,))["n"][0]
    print(f"{name!r}: {n} wheel rows")

print()
# And: restricting to just the 50 confirmed gone->done rename pairs, does that account
# for literally all 2,020 colliding filenames found earlier?
gone_done_pairs = q("""
SELECT a.name AS gone_name, b.name AS done_name
  FROM _proj_norm a
  JOIN _proj_norm b ON a.norm_name = b.norm_name AND a.name <> b.name
 WHERE a.status = 'gone' AND b.status = 'done'
""")

total_collisions_in_gone_done_pairs = 0
for _, row in gone_done_pairs.iterrows():
    fa = set(
        q("SELECT filename FROM wheel WHERE project = ?", (row["gone_name"],))[
            "filename"
        ]
    )
    fb = set(
        q("SELECT filename FROM wheel WHERE project = ?", (row["done_name"],))[
            "filename"
        ]
    )
    total_collisions_in_gone_done_pairs += len(fa & fb)

print(
    f"colliding filenames strictly within the 50 gone->done rename pairs: {total_collisions_in_gone_done_pairs:,}"
)
print(
    f"colliding filenames found across ALL 53 dupe groups earlier:       {int(total_colliding):,}"
)

'pymupdfpro': 0 wheel rows
'typhoon-hil-api': 0 wheel rows
'typhoontest': 0 wheel rows

colliding filenames strictly within the 50 gone->done rename pairs: 2,020
colliding filenames found across ALL 53 dupe groups earlier:       2,020


### Result: blanket-dropping `status='gone'` is the wrong fix -- it's both too broad and imprecise

`v.db.project` has **248** rows with `status='gone'` total. Breaking those down by whether
they have a PEP 503-normalized twin at all:

| category | count | safe to drop? |
|---|---|---|
| `gone` row with a `done` twin (genuine resolved rename -- e.g. `APICORE-Python`) | **50** | yes -- this is the actual bug |
| `gone` row whose twin has `status IS NULL` (twin discovered in the index but **never yet crawled**, 0 wheel rows) | **3** | **no** -- dropping these would delete the *only* wheel data that exists for that project today |
| `gone` row with **no twin at all** (project genuinely deleted from PyPI, never renamed) | **195** | no -- unrelated to this bug, dropping destroys real historical data for no reason |

So only 50 of the 248 `gone` rows (20%) are actually the rename-duplicate case this
investigation is about. The other 80% are either:
- legitimately-deleted, non-duplicated projects (195, the vast majority) -- blanket-dropping
  would silently erase real historical data that was never part of any conflict, or
- a twin still mid-crawl with zero wheel data of its own (3) -- blanket-dropping would delete
  the *only* copy of that project's wheels, backwards from the intended fix.

Confirmed precisely: restricting to just the 50 `gone`->`done` rename pairs accounts for
**all 2,020** colliding filenames found earlier across all 53 dupe-spelling groups -- nothing
is lost by scoping the fix that tightly, and nothing is gained by going broader.

**The correct condition is not `status = 'gone'` but "this project row's PEP 503-normalized
name has a *different* project row with `status = 'done'`"** -- i.e. compute
`pep503_normalize(project)` in the migration and prefer the `done`-status spelling whenever
two raw names collide, rather than keying anything off `status='gone'` directly.

## Verifying the chosen fix: skip `status='gone'` projects in `db2_backfill.py`

Per-conversation decision: for this one-off backfill, skip every `wheel`/`wheel_metadata`
row whose `project` is `status='gone'` in `v.db.project` (blunt, corpus-wide -- not scoped to
only the confirmed rename pairs). Accepted trade-off: the ~195 standalone `gone` projects
(never renamed, no twin) also get dropped from this migration, and the 3 `gone`+not-yet-crawled-twin
cases lose their only wheel data for now. Both are deemed acceptable for a one-off backfill of
a corpus that will be re-crawled properly later; the real fix (recognizing raw-name renames in
the crawler itself) is deferred to `crawl.py` against the new `db2` corpus.

Sanity-check the fix directly against `v.db` with the exact filtered query now in
`db2_backfill.py`.

In [21]:
# Do the collision check in SQL (GROUP BY ... HAVING) rather than pulling all ~11M
# wheel rows into pandas -- much cheaper.
before_count = q("SELECT count(*) AS n FROM wheel")["n"][0]

after_count = q("""
SELECT count(*) AS n FROM wheel
 WHERE NOT EXISTS (
     SELECT 1 FROM project p WHERE p.name = wheel.project AND p.status = 'gone'
 )
""")["n"][0]

remaining_collisions = q("""
SELECT filename, count(*) AS n
  FROM wheel
 WHERE NOT EXISTS (
     SELECT 1 FROM project p WHERE p.name = wheel.project AND p.status = 'gone'
 )
 GROUP BY filename
HAVING count(*) > 1
""")

print(f"wheel rows before filter: {before_count:,}")
print(
    f"wheel rows after filter:  {after_count:,}  (dropped {before_count - after_count:,})"
)
print(
    f"filenames still colliding after the status='gone' filter: {len(remaining_collisions)}"
)

apicore_check = q("""
SELECT project, filename
  FROM wheel
 WHERE filename IN ('apicore_python-1.0.0-py3-none-any.whl', 'apicore_python-1.0.1-py3-none-any.whl')
   AND NOT EXISTS (
       SELECT 1 FROM project p WHERE p.name = wheel.project AND p.status = 'gone'
   )
""")
apicore_check

wheel rows before filter: 12,121,854
wheel rows after filter:  12,118,369  (dropped 3,485)
filenames still colliding after the status='gone' filter: 0


,project,filename
0,APICORE_Python,apicore_python-1.0.0-py3-none-any.whl
1,APICORE_Python,apicore_python-1.0.1-py3-none-any.whl


### Verified

- **0 colliding filenames remain** anywhere in `v.db.wheel` (12,121,854 rows) after applying
  the `status='gone'` project filter -- confirms the earlier finding that all 2,020 known
  collisions were fully covered by this filter, corpus-wide.
- Only **3,485 wheel rows dropped** total (out of 12.1M) -- consistent with the ~198 `gone`
  projects' wheel rows found earlier, scaled up by however many releases each of those had.
- `apicore_python-1.0.0`/`-1.0.1` now resolve to exactly one row each, correctly kept under
  `APICORE_Python` (the live, `status='done'` spelling).

`db2_backfill.py` has been updated: `_WHEEL_SELECT` and `_WHEEL_METADATA_SELECT` now exclude
any row whose `project` is `status='gone'` in `v.db.project` (see `_SKIP_GONE_PROJECT`/
`_SKIP_GONE_PROJECT_WM`). This is documented in the module docstring as a deliberate,
corpus-wide interim shortcut for this one-off migration -- not a general-purpose rule. The
precise fix (recognize a raw-name change as a rename, not a new project) is deferred to
`crawl.py` for the new `db2` corpus's own crawl going forward.